# Digital Wayfinder Placement (Stage 4)

Given the route-legibility baseline (notebook 06) and its sensitivity analysis, this
notebook chooses **where to place digital wayfinders** to maximise legibility gain per
pound, then models the **before/after** effect.

Method:
- **Demand** = the decision points (confusion nodes) on the five inbound routes, weighted
  by how many routes use them, junction complexity, and major-road exposure.
- **Placement** = greedy maximal-coverage selection (1 - 1/e optimal, fully explainable),
  with a 150 m minimum spacing between wayfinders.
- **Effect model** = a wayfinder resolves the "which way?" ambiguity at the decision points
  it covers, improving the *intersection-complexity* and *continuity* components only.
  It does **not** change directness or crossing burden - signage cannot move roads or fix
  the Dartmouth barrier. That honesty is the point.

Descriptive only. Costs are explicit placeholder assumptions, not procurement figures.

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, networkx as nx, osmnx as ox
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
PHASE1_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == "notebooks" else PROJECT_ROOT / "phase1_spinelens_ai"
sys.path.insert(0, str(PHASE1_ROOT / "src"))
from spinelens.spatial import audit
from spinelens.metrics import legibility as lg
from spinelens.models import wayfinding as wf

DATA = PHASE1_ROOT / "data"
FIG_DIR = PHASE1_ROOT / "outputs" / "reports" / "wayfinder_media"
TABLES = PHASE1_ROOT / "outputs" / "tables"
REPORTS = PHASE1_ROOT / "outputs" / "reports"
for d in (FIG_DIR, TABLES, REPORTS):
    d.mkdir(parents=True, exist_ok=True)

# --- Placeholder cost assumptions (NOT procurement figures) ---
UNIT_COST_GBP = 6000          # installed digital wayfinder marker
WAYFINDING_BUDGET_GBP = 90000 # indicative wayfinding slice of the GBP 1,000,000 package
COVER_RADIUS_M = 150          # a wayfinder helps decisions within this distance
SPACING_M = wf.MIN_WAYFINDER_SPACING_METERS  # 150 m minimum spacing

G = ox.load_graphml(DATA / "raw" / "osm_network" / "gate0b_osm_walk_graph.graphml")
und = G.to_undirected(); largest = max(nx.connected_components(und), key=len)
coords = {n: (float(d["y"]), float(d["x"])) for n, d in G.nodes(data=True)}
cl = {n: c for n, c in coords.items() if n in largest}
import ast
def street_count(n):
    v = G.nodes[n].get("street_count")
    try: return int(float(v))
    except (TypeError, ValueError): return und.degree(n)
def _hw(h):
    if isinstance(h, list): return h
    if isinstance(h, str) and h.startswith("["):
        try: return ast.literal_eval(h)
        except (ValueError, SyntaxError): return [h]
    return [h]
node_severity = {n: 0.0 for n in G.nodes}
for u, v, d in G.edges(data=True):
    s = max((lg.crossing_severity(x) for x in _hw(d.get("highway"))), default=0.0)
    if s: node_severity[u] = max(node_severity[u], s); node_severity[v] = max(node_severity[v], s)

nodes = pd.read_csv(DATA / "route_nodes_phase1.csv").set_index("node_id")
fams = pd.read_csv(DATA / "route_families_phase1.csv")
inbound = fams[fams["origin_node_id"] != fams["gateway_node_id"]]
def snap(nid): return audit.nearest_node(cl, (float(nodes.loc[nid,"latitude"]), float(nodes.loc[nid,"longitude"])))[0]
routes = {f.route_family_id: nx.shortest_path(G, snap(f.origin_node_id), snap(f.gateway_node_id), weight="length")
          for _, f in inbound.iterrows()}
g_node = snap("ryder_street_pavilion_search_area")
print(f"routes={len(routes)} | cover_radius={COVER_RADIUS_M}m | spacing={SPACING_M}m | unit_cost=GBP{UNIT_COST_GBP}")

## Demand model: decision points needing legibility help

In [ ]:
from collections import defaultdict
route_decisions = {}          # route -> list of interior decision nodes
node_routes = defaultdict(set)
for fid, path in routes.items():
    dn = [n for n in path[1:-1] if street_count(n) >= 3]
    route_decisions[fid] = dn
    for n in dn:
        node_routes[n].add(fid)

demand_nodes = sorted(node_routes, key=lambda n: -len(node_routes[n]))
demand_weight = {}
for n in demand_nodes:
    routes_using = len(node_routes[n])
    complexity = 1 + 0.25 * (street_count(n) - 3)
    hostility = 1 + 0.30 * node_severity[n]
    demand_weight[n] = routes_using * complexity * hostility

# A candidate wayfinder at node c covers demand nodes within COVER_RADIUS_M.
covers = {c: {d for d in demand_nodes if audit.haversine_m(coords[c], coords[d]) <= COVER_RADIUS_M}
          for c in demand_nodes}

demand_df = pd.DataFrame([
    {"node": n, "lat": coords[n][0], "lon": coords[n][1], "routes_using": len(node_routes[n]),
     "street_count": street_count(n), "major_road_severity": node_severity[n],
     "demand_weight": round(demand_weight[n], 2)}
    for n in demand_nodes
]).sort_values("demand_weight", ascending=False)
print(f"decision (demand) nodes: {len(demand_nodes)} | total demand weight: {sum(demand_weight.values()):.1f}")
display(demand_df.head(10))

## Greedy placement (150 m spacing, budget-aware)

In [ ]:
too_close = lambda a, b: audit.haversine_m(coords[a], coords[b]) < SPACING_M
K_MAX = 12
picks = wf.greedy_max_coverage(demand_nodes, covers, demand_weight, K_MAX, too_close=too_close)
total_demand = sum(demand_weight.values())

curve = [p["cumulative_gain"] for p in picks]
# Recommended K = fewest wayfinders covering >= 90% of weighted demand (budget permitting).
k_budget = WAYFINDING_BUDGET_GBP // UNIT_COST_GBP
k_cover = next((i + 1 for i, c in enumerate(curve) if c >= 0.9 * total_demand), len(picks))
K = min(k_cover, k_budget, len(picks))

selected = picks[:K]
sel_nodes = [p["node"] for p in selected]
print(f"greedy found {len(picks)} useful sites; budget allows {k_budget}; 90% coverage at {k_cover}.")
print(f"Recommended K = {K} wayfinders | cost = GBP{K*UNIT_COST_GBP:,} | "
      f"covers {curve[K-1]/total_demand:.0%} of weighted demand.")

In [ ]:
def intervention_type(n):
    if node_severity[n] >= 3:        return "crossing_support"
    if street_count(n) >= 4:         return "directional_totem"
    if node_severity[n] > 0:         return "lighting_marker"
    return "ground_graphic"

place_rows = []
for rank, p in enumerate(selected, 1):
    n = p["node"]
    served = sorted({f for d in covers[n] for f in node_routes[d]})
    place_rows.append({
        "rank": rank, "node": n, "lat": round(coords[n][0], 6), "lon": round(coords[n][1], 6),
        "intervention_type": intervention_type(n),
        "street_count": street_count(n), "major_road_severity": node_severity[n],
        "routes_served": ",".join(s.replace("_to_ryder_gateway", "") for s in served),
        "marginal_gain": p["marginal_gain"], "est_cost_gbp": UNIT_COST_GBP,
    })
placement = pd.DataFrame(place_rows)
placement.to_csv(TABLES / "wayfinder_placement.csv", index=False)
print("saved:", (TABLES / "wayfinder_placement.csv").relative_to(PHASE1_ROOT))
display(placement)

## Before / after legibility

In [ ]:
covered_demand = set().union(*[covers[n] for n in sel_nodes]) if sel_nodes else set()

def route_components(fid, resolved):
    path = routes[fid]; pc = [coords[n] for n in path]
    km = nx.shortest_path_length(G, path[0], path[-1], weight="length") / 1000.0
    interior = path[1:-1]
    decisions = [n for n in interior if street_count(n) >= 3]
    dec_before = len(decisions)
    dec_after = sum(1 for n in decisions if n not in resolved)
    sig_before = sum(1 for a in lg.turn_angles(pc) if a >= lg.SIGNIFICANT_TURN_DEG)
    frac_resolved = 0.0 if dec_before == 0 else (dec_before - dec_after) / dec_before
    sig_after = sig_before * (1 - frac_resolved)
    straight = audit.haversine_m(coords[path[0]], coords[path[-1]])
    network_m = km * 1000
    severity = sum(node_severity[n] for n in interior)
    base = {
        "directness": lg.directness_score(straight, network_m),
        "turn_burden": lg.turn_burden_score(lg.total_turning_deg(pc), km),
        "crossing_burden": lg.crossing_burden_score(severity, km),
    }
    before = dict(base,
        intersection_complexity=lg.intersection_complexity_score(dec_before, km),
        continuity=lg._clamp01(1 - (sig_before / km) / lg.CONTINUITY_REF_TURNS_PER_KM))
    after = dict(base,
        intersection_complexity=lg.intersection_complexity_score(dec_after, km),
        continuity=lg._clamp01(1 - (sig_after / km) / lg.CONTINUITY_REF_TURNS_PER_KM))
    return lg.weighted_legibility_score(before), lg.weighted_legibility_score(after)

rows = []
for fid in routes:
    b, a = route_components(fid, covered_demand)
    rows.append({"route_family": fid.replace("_to_ryder_gateway", ""),
                 "RLI_before": round(b, 3), "RLI_after": round(a, 3), "delta": round(a - b, 3)})
ba = pd.DataFrame(rows).sort_values("RLI_before").reset_index(drop=True)
total_gain = ba["delta"].sum()
cost = K * UNIT_COST_GBP
ba.to_csv(TABLES / "wayfinder_before_after_rli.csv", index=False)
display(ba)
print(f"Total RLI gain across routes: {total_gain:.3f} | cost GBP{cost:,} | "
      f"clarity per GBP1000: {1000*total_gain/cost:.3f} RLI-points")

## Visuals

In [ ]:
edges = ox.graph_to_gdfs(G, nodes=False)
edges_m = edges.to_crs(27700)
def to_m(n):
    import geopandas as gpd
    from shapely.geometry import Point
    return gpd.GeoSeries([Point(coords[n][1], coords[n][0])], crs=4326).to_crs(27700).iloc[0]

fig, ax = plt.subplots(figsize=(11, 9))
edges_m.plot(ax=ax, color="#e6e6e6", linewidth=0.4, zorder=1)
for fid, path in routes.items():
    pts = [to_m(n) for n in path]
    ax.plot([p.x for p in pts], [p.y for p in pts], color="#bdbdbd", linewidth=1.3, zorder=2)
# demand nodes sized by weight
for n in demand_nodes:
    p = to_m(n)
    ax.scatter(p.x, p.y, s=8 + demand_weight[n] * 4, color="#9aa0a6", alpha=0.5, zorder=3)
# selected wayfinders + coverage circles
for p in selected:
    pm = to_m(p["node"])
    ax.add_patch(plt.Circle((pm.x, pm.y), COVER_RADIUS_M, color="#f59e0b", alpha=0.12, zorder=2))
    ax.scatter(pm.x, pm.y, marker="*", s=320, color="#f59e0b", edgecolor="black", linewidth=0.6, zorder=6)
gm = to_m(g_node)
ax.scatter(gm.x, gm.y, marker="s", s=120, color="black", edgecolor="white", zorder=7, label="Ryder gateway")
ax.set_aspect("equal"); ax.legend(loc="upper left")
ax.set_title(f"Recommended {K} digital wayfinders (amber) over decision-point demand")
fig.savefig(FIG_DIR / "figG_placement_map.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
xs = range(1, len(curve) + 1)
axes[0].plot(xs, [c / total_demand for c in curve], marker="o", color="#1a73e8")
axes[0].axvline(K, color="#f59e0b", ls="--", label=f"recommended K={K}")
axes[0].axhline(0.9, color="grey", ls=":", lw=1, label="90% demand")
axes[0].set_xlabel("number of wayfinders"); axes[0].set_ylabel("fraction of weighted demand covered")
axes[0].set_title("Diminishing returns of wayfinder coverage"); axes[0].legend(); axes[0].set_ylim(0, 1)

w = 0.38; y = np.arange(len(ba))
axes[1].barh(y - w/2, ba["RLI_before"], height=w, color="#9aa0a6", label="before")
axes[1].barh(y + w/2, ba["RLI_after"], height=w, color="#1a73e8", label="after")
axes[1].set_yticks(y); axes[1].set_yticklabels(ba["route_family"]); axes[1].invert_yaxis()
axes[1].set_xlim(0, 1); axes[1].set_xlabel("Route Legibility Index")
axes[1].set_title("Before vs after wayfinding"); axes[1].legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "figH_coverage_and_beforeafter.png", dpi=130, bbox_inches="tight"); plt.show()

## Findings note

In [ ]:
best_gain = ba.loc[ba["delta"].idxmax()]
least_gain = ba.loc[ba["delta"].idxmin()]
lines = [
    "# Wayfinder Placement Note (Stage 4)",
    "",
    "Greedy maximal-coverage placement on the Gate 0B audited network. Descriptive only.",
    "Costs are explicit placeholder assumptions, not procurement figures.",
    "",
    "## Assumptions",
    "",
    f"- Unit cost: GBP{UNIT_COST_GBP:,} per installed digital wayfinder.",
    f"- Indicative wayfinding budget: GBP{WAYFINDING_BUDGET_GBP:,} (slice of the GBP 1,000,000 package).",
    f"- Coverage radius: {COVER_RADIUS_M} m; minimum spacing: {SPACING_M} m.",
    "",
    "## Recommendation",
    "",
    f"- Place **{K} wayfinders** (cost GBP{K*UNIT_COST_GBP:,}), covering "
    f"{curve[K-1]/total_demand:.0%} of weighted decision-point demand.",
    f"- Total modelled RLI gain across the five inbound routes: {total_gain:.3f}.",
    f"- Clarity per GBP1,000: {1000*total_gain/(K*UNIT_COST_GBP):.3f} RLI-points.",
    "",
    "## Where the gain lands",
    "",
    f"- Largest legibility gain: {best_gain['route_family']} (+{best_gain['delta']:.3f}).",
    f"- Smallest legibility gain: {least_gain['route_family']} (+{least_gain['delta']:.3f}).",
    "- Wayfinding improves intersection-complexity and continuity only. Routes whose weakness",
    "  is the barrier crossing (Nechells/Dartmouth) or directness gain little from signage and",
    "  need physical crossing/public-realm intervention instead.",
    "",
    "## Selected wayfinder sites",
    "",
    "| Rank | Type | Routes served | Lat | Lon |",
    "|---:|---|---|---:|---:|",
]
for _, r in placement.iterrows():
    lines.append(f"| {r['rank']} | {r['intervention_type']} | {r['routes_served']} | {r['lat']} | {r['lon']} |")
lines += [
    "",
    "## Caveats",
    "",
    "- The before/after model is a transparent hypothesis: a wayfinder resolves the decision",
    "  points it covers. It is not a measured behavioural outcome.",
    "- Costs, coverage radius, and effect size are assumptions for sensitivity testing.",
    "- Anchors are provisional; OSM is volunteered data pending OS cross-check; no funding claim.",
]
note = "\n".join(lines)
(REPORTS / "wayfinder_placement_note.md").write_text(note, encoding="utf-8")
print(note)

## Nechells-side wayfinding - funnel the whole area to B-KQ via the barrier

The placement above covers the **city-core** approaches to the Ryder Street gateway. The
**Nechells** side is different: there is no pavilion, and the entire area approaches B-KQ
through one pinch point - the agreed **Dartmouth / Jennens crossing** - which then leads
along Jennens Road into the university cluster (Aston, BCU, Millennium Point, STEAMhouse,
Innovation Birmingham). That cluster is the entry point to B-KQ from this side. Per the
project decision there is **no tactical corridor** here (the corridor is reserved for the
city-core spine), so this is **wayfinding only**.

Modelled in two evidence-based passes on the real OSM walk network, same greedy
maximal-coverage method (150 m spacing):

- **Pass A - funnel:** every walk node in the Nechells catchment routes to the barrier;
  wayfinders are placed at the decision points that the most journeys share, so the whole
  area is directed to the crossing (the highest-reach node is the Nechells gateway totem).
- **Pass B - into the cluster:** from the barrier, Jennens Road leads to each cluster
  anchor; a few markers continue the journey into B-KQ.

These wayfinders have no pavilion to fall back on, so their content carries the hub-lite
("part-pavilion") job (notebook 11). Cluster destinations are provisional anchors pending
field validation.

In [ ]:
from collections import defaultdict

# --- Nechells-side wayfinding: funnel the whole area to B-KQ via the barrier ---
# Project decision: NO tactical corridor here (the corridor is a city-core priority).
# The ENTIRE Nechells catchment needs evidence-based wayfinding that (A) funnels the area
# to the agreed Dartmouth/Jennens crossing, then (B) leads on along Jennens Road into the
# B-KQ uni cluster (the entry point to B-KQ from this side). Same method as the inbound
# model, but placement is NEED-JUSTIFIED (not budget-filled): wider 250 m spacing (signage,
# not bus stops) and a marker is kept only if it is the gateway convergence, a *used* major-
# road crossing (severance), or a genuinely new directional - so we don't bunch markers or
# double up on the same destination. Wayfinders only; their content carries the hub-lite
# ("part-pavilion") job (no pavilion this side). Cluster anchors are provisional.
CLUSTER_DESTS = ["aston_university", "millennium_point", "bcu_parkside",
                 "steamhouse", "innovation_birmingham"]
O_SPACING = 250        # min spacing for Nechells wayfinders (wider than the 150 m city-core)
MIN_REACH = 8          # a non-severance funnel marker must serve >= this many area journeys
MIN_SEV_REACH = 3      # a severance crossing must be on a path used by >= this many journeys
bnode = snap("dartmouth_jennens_crossing")
barrier_ll = coords[bnode]
cluster_nodes = {d: snap(d) for d in CLUSTER_DESTS}
o_too_close = lambda a, b: audit.haversine_m(coords[a], coords[b]) < O_SPACING

# Nechells catchment: walk nodes on the Nechells side of the barrier (N/NE of it), within
# the study area, excluding the barrier itself and the cluster anchors.
catchment = [n for n in largest
             if coords[n][0] >= barrier_ll[0] - 0.0004 and coords[n][1] >= -1.892
             and audit.haversine_m(coords[n], barrier_ll) > 80
             and min(audit.haversine_m(coords[n], coords[cn]) for cn in cluster_nodes.values()) > 80]

# Pass A demand: every catchment journey to the barrier (the funnel).
area_routes = defaultdict(set)
for n in catchment:
    try:
        path = nx.shortest_path(G, n, bnode, weight="length")
    except nx.NetworkXNoPath:
        continue
    for m in [x for x in path[1:-1] if street_count(x) >= 3]:
        area_routes[m].add(n)
demA = sorted(area_routes, key=lambda m: -len(area_routes[m]))
wA = {m: len(area_routes[m]) * (1 + 0.25 * (street_count(m) - 3)) * (1 + 0.30 * node_severity[m]) for m in demA}
covA = {c: {d for d in demA if audit.haversine_m(coords[c], coords[d]) <= COVER_RADIUS_M} for c in demA}
picksA = wf.greedy_max_coverage(demA, covA, wA, 12, too_close=o_too_close)
# Justify: the gateway convergence + high-reach decision points + used severance crossings.
selA = [p for p in picksA
        if len(area_routes[p["node"]]) >= MIN_REACH
        or (node_severity[p["node"]] >= 2 and len(area_routes[p["node"]]) >= MIN_SEV_REACH)]
selA_nodes = [p["node"] for p in selA]

# Pass B demand: from the barrier onward to each cluster anchor (Jennens Rd into B-KQ),
# excluding nodes within spacing of a Pass-A pick.
clust_routes = defaultdict(set)
for d, cn in cluster_nodes.items():
    path = nx.shortest_path(G, bnode, cn, weight="length")
    for m in [x for x in path[1:-1] if street_count(x) >= 3]:
        clust_routes[m].add(d)
demB = [m for m in clust_routes if all(audit.haversine_m(coords[m], coords[s]) >= O_SPACING for s in selA_nodes)]
demB = sorted(demB, key=lambda m: -len(clust_routes[m]))
wB = {m: len(clust_routes[m]) * (1 + 0.25 * (street_count(m) - 3)) * (1 + 0.30 * node_severity[m]) for m in demB}
covB = {c: {d for d in demB if audit.haversine_m(coords[c], coords[d]) <= COVER_RADIUS_M} for c in demB}
picksB = wf.greedy_max_coverage(demB, covB, wB, 6, too_close=o_too_close)
# Justify: keep severance crossings + ONE best directional per destination-group (no dupes).
selB, seen_groups = [], set()
for p in picksB:
    n = p["node"]
    group = tuple(sorted({d for dd in covB[n] for d in clust_routes[dd]}))
    if node_severity[n] >= 2 or group not in seen_groups:
        selB.append(p); seen_groups.add(group)

def nearest_anchor(n):
    return min(cluster_nodes, key=lambda d: nx.shortest_path_length(G, n, cluster_nodes[d], weight="length"))

base_rank = int(placement["rank"].max())
top_funnel = selA[0]["node"] if selA else None  # highest-reach node = the Nechells gateway totem
o_rows, rank = [], base_rank
for p in selA:
    n = p["node"]; rank += 1
    itype = "directional_totem" if n == top_funnel else intervention_type(n)
    o_rows.append({"rank": rank, "node": n, "lat": round(coords[n][0], 6), "lon": round(coords[n][1], 6),
                   "intervention_type": itype, "street_count": street_count(n),
                   "major_road_severity": node_severity[n], "serves_destinations": nearest_anchor(n),
                   "marginal_gain": p["marginal_gain"], "est_cost_gbp": UNIT_COST_GBP,
                   "leg": "nechells_to_barrier"})
for p in selB:
    n = p["node"]; rank += 1
    served = sorted({d for dd in covB[n] for d in clust_routes[dd]})
    o_rows.append({"rank": rank, "node": n, "lat": round(coords[n][0], 6), "lon": round(coords[n][1], 6),
                   "intervention_type": intervention_type(n), "street_count": street_count(n),
                   "major_road_severity": node_severity[n], "serves_destinations": ",".join(served),
                   "marginal_gain": p["marginal_gain"], "est_cost_gbp": UNIT_COST_GBP,
                   "leg": "barrier_to_cluster"})
onward_placement = pd.DataFrame(o_rows)
onward_placement.to_csv(TABLES / "wayfinder_placement_onward.csv", index=False)
print(f"Nechells catchment={len(catchment)} nodes | spacing={O_SPACING} m")
print(f"justified: Pass A (to barrier) {len(selA)} | Pass B (into cluster) {len(selB)} "
      f"-> {len(o_rows)} Nechells-side wayfinders (cost GBP{len(o_rows)*UNIT_COST_GBP:,})")
display(onward_placement)

# Visual: the catchment funnel to the barrier + onward into the cluster.
try:
    fig, ax = plt.subplots(figsize=(10, 9))
    edges_m.plot(ax=ax, color="#e6e6e6", linewidth=0.4, zorder=1)
    for n in catchment:
        pm = to_m(n); ax.scatter(pm.x, pm.y, s=3, color="#c7ccd1", alpha=0.5, zorder=2)
    for p in selA:
        pm = to_m(p["node"])
        ax.add_patch(plt.Circle((pm.x, pm.y), COVER_RADIUS_M, color="#f59e0b", alpha=0.10, zorder=2))
        ax.scatter(pm.x, pm.y, marker="*", s=300, color="#f59e0b", edgecolor="black", linewidth=0.6, zorder=6)
    for p in selB:
        pm = to_m(p["node"])
        ax.add_patch(plt.Circle((pm.x, pm.y), COVER_RADIUS_M, color="#10b981", alpha=0.10, zorder=2))
        ax.scatter(pm.x, pm.y, marker="*", s=300, color="#10b981", edgecolor="black", linewidth=0.6, zorder=6)
    bm = to_m(bnode)
    ax.scatter(bm.x, bm.y, marker="X", s=180, color="#d93025", edgecolor="white", zorder=7, label="Dartmouth/Jennens barrier")
    for d in CLUSTER_DESTS:
        dm = to_m(cluster_nodes[d])
        ax.scatter(dm.x, dm.y, marker="s", s=90, color="black", edgecolor="white", zorder=7)
        ax.annotate(d.replace("_", " "), (dm.x, dm.y), fontsize=8, xytext=(4, 4), textcoords="offset points")
    ax.scatter([], [], marker="*", s=200, color="#f59e0b", label="funnel to barrier (Pass A)")
    ax.scatter([], [], marker="*", s=200, color="#10b981", label="into cluster (Pass B)")
    ax.set_aspect("equal"); ax.legend(loc="upper left")
    ax.set_title(f"Nechells wayfinding (need-justified): {len(o_rows)} markers to the barrier and into B-KQ")
    fig.savefig(FIG_DIR / "figG2_nechells_funnel_map.png", dpi=130, bbox_inches="tight"); plt.show()
except Exception as e:
    print("onward visual skipped:", e)

## What this unlocks

A ranked, budget-aware wayfinder placement with a transparent before/after legibility
model and a clarity-per-pound figure - the first direct answer to the project's core
question. It also makes explicit that signage helps confusion and continuity, while the
Dartmouth barrier needs a physical crossing intervention (the pavilion/crossing scope).
Next: tactical-corridor synthesis from the shared trunk, and a sensitivity pass on the
cost/effect assumptions.